# Update Gronings↔Dutch Translation Model v2

This notebook walks through the full pipeline for updating the
huggingFace model:

1. **Download** the latest five-language Tatoeba export and record its corpus counts
2. **Train** NLLB 1.3B for 8 epochs on Tatoeba plus synthetic Kreuze data. This notebook implements the strongest recipe found in issue #34.
3. **Evaluate** all checkpoints
4. **Inspect** sample translations
5. **Upload** to HuggingFace

Run cells top-to-bottom. You only need to set the Huggingface token in .env

## 1. Configuration

Run this notebook from the repository root. Commit the notebook setup before
starting training so the provenance manifest can record a meaningful Git revision.

In [ ]:
import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from pandas.testing import assert_frame_equal

load_dotenv(".env")

from nllb_try.corpus import (
    GLOBAL_HOLDOUT_FRACTION,
    SPLIT_SEED,
    TatoebaCorpus,
    main_corpus,
)
from nllb_try.downloadtatoeba import main_download
from nllb_try.evaluate import main_evaluate, translate
from nllb_try.tokenizer_and_model_setup import setup_model_and_tokenizer
from nllb_try.train import get_training_budget, main_train
from sidetracks.kreuze.run_kreuze_multilingual_continuation import (
    build_continuation,
)
from sidetracks.kreuze.run_kreuze_multilingual_experiment import (
    FOCUS_PAIR,
    GEMMA_CORPUS,
    NLLB_LANGS,
    TATOEBA_LANGS,
    build_experiment,
    print_plan,
)

MODELNAME = "facebook/nllb-200-distilled-1.3B"
DEVICE = "cuda:1"
SEED = 9358
BASE_EPOCHS = 8
TATOEBA_PATH = "data/tatoeba"
REPO_ID = "Tom9358/nllb-gos-nld-v2"

assert Path(GEMMA_CORPUS).is_file(), f"Missing Kreuze corpus: {GEMMA_CORPUS}"


def update_run_config(run_dir, **values):
    """Merge arbitrary metadata into a run's persisted run_config.json.

    Keys are not validated and this does not modify the active RunConfig or
    affect training behavior. Call it only after main_train() has written the
    run configuration.
    """
    path = Path(run_dir) / "run_config.json"
    config = json.loads(path.read_text(encoding="utf-8"))
    config.update(values)
    path.write_text(
        json.dumps(config, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )


def print_budget(label, corpora, cfg):
    budget = get_training_budget(corpora, cfg)
    print(f"\n{label}")
    print(f"  samples/epoch: {budget['samples_per_epoch']:,}")
    print(f"  steps/epoch:   {budget['steps_per_epoch']:,}")
    print(f"  total steps:   {budget['total_steps']:,}")
    return budget


def assert_same_tatoeba_splits(reference, candidate, *, compare_train):
    """Fail if two corpus builds do not contain identical Tatoeba splits."""
    def by_pair(corpora):
        return {
            (corpus.source_lang_nllb, corpus.target_lang_nllb): corpus
            for corpus in corpora
            if isinstance(corpus, TatoebaCorpus)
        }

    reference_by_pair = by_pair(reference)
    candidate_by_pair = by_pair(candidate)
    assert reference_by_pair.keys() == candidate_by_pair.keys()
    for pair, reference_corpus in reference_by_pair.items():
        candidate_corpus = candidate_by_pair[pair]
        assert_frame_equal(
            reference_corpus.df_validate,
            candidate_corpus.df_validate,
            obj=f"{pair} validation split",
        )
        if compare_train:
            assert_frame_equal(
                reference_corpus.df_train,
                candidate_corpus.df_train,
                obj=f"{pair} training split",
            )

## 2. Download Data and Build the Pooled Stage

The Tatoeba export is mutable and is not archived here. The notebook records
the download time, corpus counts, split settings, Git revision, and exact run
configuration. Kreuze is pooled only into the Dutch-Gronings training corpus;
its own validation split remains separate for evaluation.

In [ ]:
main_download(TATOEBA_LANGS, redownload=True, tatoeba_path=TATOEBA_PATH)
downloaded_at = datetime.now(timezone.utc).isoformat()

base_corpora, base_cfg = build_experiment(
    variant="pooled",
    device=DEVICE,
    modelname=MODELNAME,
    seed=SEED,
    num_epochs=BASE_EPOCHS,
)
# Just some sanity checks so I really know how I'm training
assert base_cfg.sampling_strategy == "focus_total"
assert base_cfg.focus_lang_pair == FOCUS_PAIR
assert base_cfg.direction_strategy == "alternating"
assert base_cfg.num_epochs == 8
print_plan("v2 pooled base", base_corpora, base_cfg)

# Build an independent unpooled view for evaluation and corpus-count provenance.
evaluation_corpora = main_corpus(
    source_langs_tatoeba=TATOEBA_LANGS,
    source_langs_nllb=NLLB_LANGS,
    parallel_data_paths=(GEMMA_CORPUS,),
    cfg=base_cfg,
)
# Pooling changes the Stage 1 focus training rows, but validation must remain
# byte-for-byte identical to the independent evaluation view.
assert_same_tatoeba_splits(
    base_corpora,
    evaluation_corpora,
    compare_train=False,
)
print("Confirmed identical Stage 1 and evaluation Tatoeba validation splits.")

git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
corpus_counts = []
for corpus in evaluation_corpora:
    corpus_counts.append({
        "kind": "tatoeba" if isinstance(corpus, TatoebaCorpus) else "parallel",
        "source_lang_nllb": corpus.source_lang_nllb,
        "target_lang_nllb": corpus.target_lang_nllb,
        "path": getattr(corpus, "path", None),
        "total_rows": len(corpus.df_train) + len(corpus.df_validate),
        "train_rows": len(corpus.df_train),
        "validation_rows": len(corpus.df_validate),
    })

provenance = {
    "release": "v2",
    "downloaded_at": downloaded_at,
    "tatoeba_source": "https://tatoeba.org/en/downloads",
    "tatoeba_snapshot_archived": False,
    "split_strategy": "global_sentence_id_holdout",
    "split_seed": SPLIT_SEED,
    "holdout_fraction": GLOBAL_HOLDOUT_FRACTION,
    "git_commit": git_commit,
    "kreuze_corpus": GEMMA_CORPUS,
    "corpora": corpus_counts,
}
base_run_dir = Path(base_cfg.run_dir)
base_run_dir.mkdir(parents=True, exist_ok=True)
provenance_path = base_run_dir / "training_data_manifest.json"
provenance_path.write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(f"Saved data provenance to {provenance_path}")

## 3. Train Eight Pooled Epochs

Each epoch is exactly 50% pooled Dutch-Gronings and 50% supporting-language
Tatoeba data distributed with temperature `T=5`.

In [ ]:
base_budget = print_budget("Pooled base budget", base_corpora, base_cfg)
main_train(base_corpora, base_cfg)
update_run_config(
    base_cfg.run_dir,
    experiment="kreuze-phase2-pooled",
    release_recipe="v2-pooled-eight-epochs-plus-clean-finish",
    training_data_manifest=str(provenance_path),
)
print(f"Pooled stage complete: {base_cfg.run_dir}")

## 4. Clean Tatoeba Finish

The continuation starts from pooled epoch 8 and contains no Kreuze samples.
Its focus half is one complete Dutch-Gronings training pass; the other half is
temperature-balanced supporting Tatoeba data. Logical epoch 8 reverses every
stable focus row relative to pooled epoch 8's logical epoch 7.

In [ ]:
clean_corpora, clean_cfg = build_continuation(
    variant="clean",
    source_run=Path(base_cfg.run_dir),
    device=DEVICE,
    source_epoch=BASE_EPOCHS,
)
assert clean_cfg.initial_model_path == str(
    Path(base_cfg.run_dir) / "checkpoints" / "epoch8"
)
assert clean_cfg.num_epochs == 1
assert clean_cfg.direction_strategy == "alternating"
assert clean_cfg.direction_epoch_offset == 8
# build_continuation reconstructs the corpora from the local files. Stop before
# training if those files changed and produced different train/validation rows.
assert_same_tatoeba_splits(
    evaluation_corpora,
    clean_corpora,
    compare_train=True,
)
print("Confirmed identical Tatoeba train/validation splits for both stages.")
clean_budget = print_budget("Clean-finish budget", clean_corpora, clean_cfg)

main_train(clean_corpora, clean_cfg)
clean_run_dir = Path(clean_cfg.run_dir)
clean_manifest = clean_run_dir / "training_data_manifest.json"
shutil.copy2(provenance_path, clean_manifest)
update_run_config(
    clean_cfg.run_dir,
    experiment="kreuze-phase2b-clean",
    release_recipe="v2-pooled-eight-epochs-plus-clean-finish",
    source_run=str(base_cfg.run_dir),
    source_epoch=BASE_EPOCHS,
    training_data_manifest=str(clean_manifest),
)
print(f"Clean finish complete: {clean_cfg.run_dir}")

## 5. Evaluate the Final v2 Checkpoint

This evaluates the clean-finished `epoch1` checkpoint on the complete fixed
validation splits produced earlier in this notebook. It does not evaluate the
pretrained baseline because the meaningful baseline for a continuation would
be pooled epoch 8, not the original Facebook model.

In [ ]:
main_evaluate(
    corpus_objects=evaluation_corpora,
    run_dir=clean_cfg.run_dir,
    new_lang_nllb=clean_cfg.new_lang_nllb,
    device=clean_cfg.device,
    sample_size=None,
    include_baseline=False,
    include_train=False,
    verbose=True,
)
FINAL_MODEL_PATH = Path(clean_cfg.run_dir) / "checkpoints" / "epoch1"
assert FINAL_MODEL_PATH.is_dir(), FINAL_MODEL_PATH
print(f"Final v2 checkpoint: {FINAL_MODEL_PATH}")

## 6. Inspect Translations

In [ ]:
model, tokenizer = setup_model_and_tokenizer(
    str(FINAL_MODEL_PATH),
    modelpath="hfacemodels",
    new_lang="gos_Latn",
    device=DEVICE,
)

sentences_nld = [
    "Ik ga morgen naar de stad.",
    "Het regent buiten. Neem een paraplu mee!",
    "De kat slaapt zachtjes op de bank.",
    "De man die je gisteren zag, is mijn buurman.",
    "Het kind dat in de speeltuin speelt, is mijn neefje.",
    "Achter de wolken schijnt de zon.",
    "De trein vertrekt om 15:30 uur vanaf perron 5.",
    "Ik heb een nieuwe fiets gekocht en hij rijdt geweldig.",
    "Het boek dat ik lees, is erg spannend en moeilijk om neer te leggen.",
    "De bloemen in de tuin bloeien prachtig in de lente.",
    "Het vliegtuig landt op tijd op de luchthaven.",
    "De hond blaft naar de postbode die langs het huis loopt.",
    "Ik heb een afspraak bij de tandarts om 10 uur.",
    "Het restaurant serveert heerlijke Italiaanse gerechten.",
    "Ik heb een nieuwe baan gevonden en begin volgende week.",
    "Het tuinhuis in de ruimte ruikt naar verse verf en hout.",
]

print("\n--- Nederlands -> Gronings ---")
translations_gos = []
for sentence in sentences_nld:
    translated = translate(
        sentence,
        src_lang="nld_Latn",
        tgt_lang="gos_Latn",
        model=model,
        tokenizer=tokenizer,
    )
    translations_gos.append(translated)
    print(f"NL:  {sentence}")
    print(f"GOS: {translated}\n")

print("--- Gronings -> Nederlands ---")
for sentence in translations_gos:
    translated = translate(
        sentence,
        src_lang="gos_Latn",
        tgt_lang="nld_Latn",
        model=model,
        tokenizer=tokenizer,
    )
    print(f"GOS: {sentence}")
    print(f"NL:  {translated}\n")

## 7. Upload to Hugging Face

Set `HF_TOKEN` in `.env`. Upload is deliberately blocked until
`CONFIRM_UPLOAD` is changed to `True`. This repository is separate from the
Tatoeba-only v1 model.

In [ ]:
import huggingface_hub

CONFIRM_UPLOAD = False
assert CONFIRM_UPLOAD, "Inspect evaluation and translations before uploading."
assert REPO_ID != "Tom9358/nllb-tatoeba-gos-nld-v1"
assert FINAL_MODEL_PATH.is_dir(), FINAL_MODEL_PATH

huggingface_hub.login(token=os.environ["HF_TOKEN"])
upload_model, upload_tokenizer = setup_model_and_tokenizer(
    str(FINAL_MODEL_PATH),
    modelpath="hfacemodels",
    new_lang="gos_Latn",
    device=DEVICE,
)
upload_tokenizer.push_to_hub(
    REPO_ID,
    commit_message="Upload NLLB Gronings v2 tokenizer",
    private=True,
)
upload_model.push_to_hub(
    REPO_ID,
    commit_message="Upload NLLB Gronings v2 model",
    private=True,
)